# Yaya-125M — Kaggle Training

**Before running:** Settings → Accelerator → **GPU T4 x2** (or P100) → Save

## Session flow
| Session | What happens |
|---------|-------------|
| 1st | Tokenize Wikipedia (~15 min) + pretrain 5000 steps (~10 hrs) |
| 2nd+ | Download checkpoint dataset → auto-resume pretrain or start SFT |

## Checkpoint persistence between sessions
Kaggle doesn't have persistent storage like Drive. Between sessions:
1. After each session ends, go to **Output tab** → download `yaya-checkpoints.zip`
2. Upload it as a **Kaggle Dataset** (once)
3. On the next session, attach that dataset → checkpoint is auto-detected

Or skip persistence and just train start-to-finish in one ~12h session (T4 x2 is faster).


## Step 0 — Check GPU & setup

In [ ]:
import torch, os, sys, glob, shutil

if torch.cuda.is_available():
    name = torch.cuda.get_device_name(0)
    vram = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
    print(f'GPU : {name}  ({vram} GB VRAM)')
    print(f'GPU count: {torch.cuda.device_count()}')
    print(f'PyTorch: {torch.__version__}')
    DTYPE = 'bfloat16' if torch.cuda.is_bf16_supported() else 'float16'
    IS_A100 = 'A100' in name
    IS_DUAL = torch.cuda.device_count() >= 2
else:
    raise RuntimeError('No GPU found. Go to Settings → Accelerator → GPU T4 x2')

print(f'dtype: {DTYPE}  |  A100: {IS_A100}  |  dual GPU: {IS_DUAL}')

# Working dirs
WORK_DIR     = '/kaggle/working'
CKPT_DIR     = f'{WORK_DIR}/yaya-checkpoints'
PRETRAIN_DIR = f'{CKPT_DIR}/pretrain'
SFT_DIR      = f'{CKPT_DIR}/sft'
DATA_CACHE   = f'{WORK_DIR}/data_cache'
WIKI_TRAIN   = f'{DATA_CACHE}/wiki_train'
WIKI_EVAL    = f'{DATA_CACHE}/wiki_eval'

for d in [PRETRAIN_DIR, SFT_DIR, DATA_CACHE, WIKI_TRAIN, WIKI_EVAL]:
    os.makedirs(d, exist_ok=True)

print('Dirs ready.')

In [ ]:
import subprocess

# Clone or update repo
if not os.path.exists('/kaggle/working/miss-yaya'):
    subprocess.run(['git', 'clone', 'https://github.com/jaylink-coder/miss-yaya.git',
                    '/kaggle/working/miss-yaya'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/miss-yaya', 'pull'], check=True)

# Install deps
subprocess.run(['pip', 'install', '-q', 'sentencepiece', 'pyyaml', 'datasets'], check=True)

os.chdir('/kaggle/working/miss-yaya/yaya-ai')
sys.path.insert(0, '/kaggle/working/miss-yaya/yaya-ai')
print('Ready:', os.getcwd())

# Restore checkpoint from attached Kaggle dataset (if any)
# Attach your saved dataset as /kaggle/input/yaya-checkpoints before running
ATTACHED_CKPT = '/kaggle/input/yaya-checkpoints'
if os.path.exists(ATTACHED_CKPT):
    print(f'Restoring checkpoints from attached dataset: {ATTACHED_CKPT}')
    shutil.copytree(ATTACHED_CKPT, CKPT_DIR, dirs_exist_ok=True)
    pretrain_ckpts = sorted(glob.glob(f'{PRETRAIN_DIR}/checkpoint-*'))
    sft_ckpts = sorted(glob.glob(f'{SFT_DIR}/checkpoint-*'))
    print(f'  Pretrain checkpoints: {len(pretrain_ckpts)}')
    print(f'  SFT checkpoints: {len(sft_ckpts)}')
else:
    print('No attached checkpoint dataset — starting fresh.')

## Phase 1 — Pretrain on Wikipedia

Gives Yaya world knowledge. **Skip if PRETRAIN_DIR already has checkpoints** (auto-detected).

In [ ]:
# Step 5: Evaluate
import glob
sft_ckpts = sorted(glob.glob('/kaggle/working/yaya-sft-1b-checkpoints/checkpoint-*'))
best = sft_ckpts[-1] if sft_ckpts else None
if best:
    import os
    os.system(
        f'python scripts/eval_yaya.py '
        f'--model_config configs/model/yaya_1b.yaml '
        f'--checkpoint {best} --chat'
    )
else:
    print('No SFT checkpoint found.')